# NIFTY Gap Strategy — v8 (Walk-Forward + Random Forest + Bootstrap CI + Kelly Sizing)

Four improvements over v7 (single-split L1 logistic regression):

| Improvement | What it fixes |
|---|---|
| **Walk-forward expanding window** | v7's single split still had threshold selected in-sample. Here: retrain + reselect threshold monthly. |
| **Random Forest comparison** | Logistic regression assumes linearity. RF (max_depth=3) finds interactions without overfitting. |
| **Bootstrap CI** | 15-month OOS = still noisy. Bootstrap tells us the probability the edge is real. |
| **Kelly criterion sizing** | Fixed `capital // cost_lot` over-bets at high capital. Kelly is theoretically optimal. |

**Walk-forward design:**
- Warmup: Jan 2024 – Dec 2024 (12 months, ~173 days)
- OOS: Jan 2025 – Mar 2026 (15 months, one month at a time)
- Each month: refit model on all prior data, auto-select threshold, predict next month only
- Compounding capital flows continuously across all 15 test months

**No re-simulation:** reuses `v6/sim_cache.csv` + `v2/v2_aligned_dataset.csv`. Runtime: ~1 min.

In [17]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Trade parameters (unchanged across all versions) ─────────────────────────
SL_PCT           = 0.15
TP_PCT           = 0.40
LOT_SIZE         = 75
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0
BREAKEVEN        = SL_PCT / (SL_PCT + TP_PCT)   # 27.3%
KELLY_ODDS       = TP_PCT / SL_PCT               # 2.667 = net win / net loss

# ── Walk-forward config ────────────────────────────────────────────────────────
WF_WARMUP_MONTHS = 12           # months of mandatory training before first OOS test
OOS_FIRST_MONTH  = (2025, 1)    # first month tested OOS

# ── L1 Logistic config (extended C grid vs v7) ────────────────────────────────
CV_C_GRID    = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
CLASS_WEIGHT = {0: 1.0, 1: 2.5}

# ── Random Forest config (shallow = low variance, Opus recommendation) ────────
RF_MAX_DEPTH        = 3
RF_MIN_SAMPLES_LEAF = 20
RF_N_ESTIMATORS     = 300

# ── Threshold selection ────────────────────────────────────────────────────────
EDGE_TARGET_PP    = 8     # target base_win_rate + 8pp on training window
MIN_THRESH_TRADES = 10    # minimum in-sample trades at threshold (smaller than v7 due to monthly windows)

# ── Degenerate model guard ────────────────────────────────────────────────────
MIN_TRAIN_AUC     = 0.53  # skip month if training AUC below this
MIN_PROB_STD      = 0.02  # skip month if prob std near-constant (L1 zeroed all features)

# ── Bootstrap CI ──────────────────────────────────────────────────────────────
N_BOOTSTRAP = 10_000

print('Config loaded.')
print(f'Walk-forward warmup : {WF_WARMUP_MONTHS} months | First OOS : {OOS_FIRST_MONTH}')
print(f'L1 C grid           : {CV_C_GRID}')
print(f'RF                  : max_depth={RF_MAX_DEPTH}, min_samples_leaf={RF_MIN_SAMPLES_LEAF}, n={RF_N_ESTIMATORS}')
print(f'Breakeven win rate  : {BREAKEVEN:.1%}  |  Kelly odds (TP/SL) : {KELLY_ODDS:.3f}')

Config loaded.
Walk-forward warmup : 12 months | First OOS : (2025, 1)
L1 C grid           : [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
RF                  : max_depth=3, min_samples_leaf=20, n=300
Breakeven win rate  : 27.3%  |  Kelly odds (TP/SL) : 2.667


In [18]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING    = Path.cwd().parent
SIM_CACHE_PATH = GAP_TRADING / 'v6' / 'sim_cache.csv'
ALIGNED_CSV    = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'

for lbl, p in [('sim_cache (v6)', SIM_CACHE_PATH), ('aligned CSV (v2)', ALIGNED_CSV)]:
    print(f'{lbl:<20}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Load + merge (identical to v7) ────────────────────────────────────────────
sim_df = pd.read_csv(SIM_CACHE_PATH, parse_dates=['date'])
sim_df['date'] = sim_df['date'].dt.date
sim_core = sim_df[['date', 'win', 'exit_reason', 'entry_prem', 'exit_prem', 'dte']].copy()

aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned['india_date'] = aligned['india_date'].dt.date
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()

# ── Rolling regime features — computed on full aligned history before merging ─
# aligned starts Mar 2023, so Jan 2024 sim_cache rows have ~200 days of lookback.
aligned = aligned.sort_values('india_date').reset_index(drop=True)
aligned['nifty_20d_realized_vol'] = aligned['prev_india_ret'].rolling(20).std()
aligned['nifty_20d_ret']          = (
    np.exp(np.log1p(aligned['prev_india_ret']).rolling(20).sum()) - 1
)

ALIGNED_COLS = ['india_date', 'gap_pct', 'prev_india_ret',
                'SP500_ret', 'NASDAQ_ret', 'DOW_ret',
                'DAX_ret', 'FTSE_ret',
                'NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret',
                'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level',
                'nifty_20d_ret', 'nifty_20d_realized_vol']

merged = (sim_core
          .merge(aligned[ALIGNED_COLS], left_on='date', right_on='india_date', how='inner')
          .drop(columns=['india_date']))

# Composite features
merged['us_ret']         = merged[['SP500_ret', 'NASDAQ_ret', 'DOW_ret']].mean(axis=1)
merged['europe_ret']     = merged[['DAX_ret', 'FTSE_ret']].mean(axis=1)
merged['asia_ret']       = merged[['NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret']].mean(axis=1)
merged['log_entry_prem'] = np.log(merged['entry_prem'].clip(lower=0.1))
merged['month_key']      = merged['date'].apply(lambda d: (d.year, d.month))

# gap expressed in units of recent volatility
merged['gap_normalized'] = merged['gap_pct'] / merged['nifty_20d_realized_vol'].replace(0, np.nan)

# ── Feature set (13 features: 10 original + 3 regime features) ───────────────
FEATURES = [
    'gap_pct', 'prev_india_ret',
    'us_ret', 'europe_ret', 'asia_ret',
    'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level',
    'log_entry_prem', 'dte',
    'nifty_20d_ret',          # 20-day NIFTY trend
    'nifty_20d_realized_vol', # 20-day realized vol
    'gap_normalized',         # gap in vol units
]

before = len(merged)
merged = merged.dropna(subset=FEATURES).reset_index(drop=True)
merged = merged.sort_values('date').reset_index(drop=True)

all_months = sorted(merged['month_key'].unique())
oos_start_idx = all_months.index(OOS_FIRST_MONTH)

print(f'\nMerged   : {len(merged)} rows  (dropped {before-len(merged)} NaN rows)')
print(f'  (extra NaNs vs v7 are from rolling 20-day window at dataset start)')
print(f'Period   : {merged["date"].min()} → {merged["date"].max()}')
print(f'Months   : {len(all_months)} total — warmup {all_months[0]}→{all_months[oos_start_idx-1]}, OOS {OOS_FIRST_MONTH}→{all_months[-1]}')
print(f'OOS test : {len(all_months) - oos_start_idx} months')
print(f'\nBase win rate (all data)  : {merged["win"].mean():.1%}')
print(f'Base win rate (OOS period): {merged[merged["month_key"] >= OOS_FIRST_MONTH]["win"].mean():.1%}')

sim_cache (v6)      : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v6\sim_cache.csv)
aligned CSV (v2)    : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)

Merged   : 382 rows  (dropped 20 NaN rows)
  (extra NaNs vs v7 are from rolling 20-day window at dataset start)
Period   : 2024-01-02 → 2026-03-24
Months   : 27 total — warmup (2024, 1)→(2024, 12), OOS (2025, 1)→(2026, 3)
OOS test : 15 months

Base win rate (all data)  : 25.1%
Base win rate (OOS period): 24.1%


In [19]:
# ── Shared helper functions ────────────────────────────────────────────────────

def round_trip_charges(ep: float, xp: float, lots: int) -> float:
    buy_val  = ep * lots * LOT_SIZE
    sell_val = xp * lots * LOT_SIZE
    brok  = 40.0
    stamp = 0.00003  * buy_val
    stt   = 0.000625 * sell_val
    exch  = 0.00053  * (buy_val + sell_val)
    sebi  = 0.000001 * (buy_val + sell_val)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)


def auto_threshold(tr_df: pd.DataFrame, base_wr: float) -> float:
    """Pick highest threshold with Win% >= base_wr + EDGE_TARGET_PP and >= MIN_THRESH_TRADES trades."""
    target = base_wr + EDGE_TARGET_PP / 100
    best   = None
    for thr in np.arange(0.24, 0.70, 0.02).round(2):
        sub = tr_df[tr_df['prob'] >= thr]
        if len(sub) >= MIN_THRESH_TRADES and sub['win'].mean() >= target:
            best = thr
    if best is None:  # fallback: lowest threshold with any positive edge + 5 trades
        for thr in np.arange(0.24, 0.70, 0.02).round(2):
            sub = tr_df[tr_df['prob'] >= thr]
            if len(sub) >= 5 and sub['win'].mean() > base_wr:
                best = thr
                break
    return best if best is not None else 0.32


def compute_ledger(oos_rows: list, use_kelly: bool = False) -> pd.DataFrame:
    """Convert list of OOS trade dicts to compounding ledger."""
    traded = [r for r in oos_rows if r['traded']]
    if not traded:
        return pd.DataFrame()
    capital, peak, rows = STARTING_CAPITAL, STARTING_CAPITAL, []
    for r in traded:
        ep, xp, dte = float(r['entry_prem']), float(r['exit_prem']), int(r['dte'])
        prob = float(r['prob'])
        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0:
            continue
        if use_kelly:
            f    = max(0.0, prob - (1 - prob) / KELLY_ODDS)  # full Kelly fraction
            lots = max(1, int(f * capital / cost_lot))
        else:
            lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0:
            lots = min(lots, DTE0_MAX_LOTS)
        chg      = round_trip_charges(ep, xp, lots)
        trade_pnl = (xp - ep) * LOT_SIZE * lots - chg
        capital  += trade_pnl
        peak      = max(peak, capital)
        rows.append({
            'Date'     : r['date'],
            'Month'    : r['month'],
            'Model'    : r['model'],
            'P(win)'   : round(prob, 3),
            'Thr'      : round(r['threshold'], 2),
            'DTE'      : dte,
            'Lots'     : lots,
            'Entry'    : ep,
            'Exit'     : xp,
            'PnL(pts)' : round(xp - ep, 2),
            'Trade PnL': round(trade_pnl, 2),
            'Capital'  : round(capital, 2),
            'DD%'      : round((peak - capital) / peak * 100, 2),
            'Reason'   : r['exit_reason'],
        })
    return pd.DataFrame(rows)


def print_ledger_summary(ledger: pd.DataFrame, label: str):
    if ledger.empty:
        print(f'\n{label}: 0 trades.')
        return
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    roi   = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['DD%'].max()
    aw    = ledger.loc[ledger['Trade PnL'] > 0,  'Trade PnL'].mean() if wins > 0     else 0.0
    al    = ledger.loc[ledger['Trade PnL'] <= 0, 'Trade PnL'].mean() if wins < total else 0.0
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'  Trades   : {total}')
    print(f'  Win rate : {wins/total*100:.1f}%  (breakeven: {BREAKEVEN:.1%})')
    print(f'  ROI      : {roi:+.1f}%')
    print(f'  Max DD   : {maxdd:.1f}%')
    print(f'  Avg win  : Rs {aw:,.0f}  |  Avg loss: Rs {al:,.0f}')
    print(f'{"="*60}')
    print(ledger['Reason'].value_counts().to_string())
    print()
    print(ledger[['Date', 'P(win)', 'Thr', 'DTE', 'Lots', 'Entry', 'Exit',
                  'PnL(pts)', 'Trade PnL', 'Capital', 'DD%', 'Reason']].to_string(index=False))


print('Helpers ready.')

Helpers ready.


In [20]:
# ── Walk-forward engine ────────────────────────────────────────────────────────
# Runs for any model_fn. Returns (oos_rows list, monthly_summary list).

def run_walk_forward(model_name: str, fit_fn):
    """
    fit_fn(Xs_train, y_train) -> fitted model with .predict_proba()
    Returns: (oos_rows, monthly_summary)
    """
    oos_rows, monthly = [], []

    for i in range(oos_start_idx, len(all_months)):
        test_month = all_months[i]
        yr, mo     = test_month

        tr = merged[merged['month_key'] < test_month].reset_index(drop=True)
        te = merged[merged['month_key'] == test_month].reset_index(drop=True)
        if len(te) == 0:
            continue

        X_tr = tr[FEATURES].values
        y_tr = tr['win'].astype(int).values
        X_te = te[FEATURES].values

        sc     = StandardScaler()
        Xs_tr  = sc.fit_transform(X_tr)
        Xs_te  = sc.transform(X_te)

        base_wr = y_tr.mean()
        model   = fit_fn(Xs_tr, y_tr)

        probs_tr = model.predict_proba(Xs_tr)[:, 1]
        probs_te = model.predict_proba(Xs_te)[:, 1]

        # ── Degenerate model guard ────────────────────────────────────────────
        train_auc  = roc_auc_score(y_tr, probs_tr)
        prob_std   = float(probs_tr.std())
        degenerate = train_auc < MIN_TRAIN_AUC or prob_std < MIN_PROB_STD

        best_C_str = f"{model.C_[0]:.2f}" if hasattr(model, 'C_') else 'N/A'

        if degenerate:
            thr = 1.0  # effectively infinite — nothing trades
            for j, (_, row) in enumerate(te.iterrows()):
                oos_rows.append({
                    'date'       : row['date'],
                    'month'      : test_month,
                    'model'      : model_name,
                    'prob'       : float(probs_te[j]),
                    'threshold'  : thr,
                    'traded'     : False,
                    'win'        : bool(row['win']),
                    'exit_reason': row['exit_reason'],
                    'entry_prem' : row['entry_prem'],
                    'exit_prem'  : row['exit_prem'],
                    'dte'        : row['dte'],
                    'base_wr'    : base_wr,
                    'VIX_INDIA_level': row['VIX_INDIA_level'],
                })
            monthly.append({
                'Month'     : f'{yr}-{mo:02d}',
                'Train days': len(tr),
                'Test days' : len(te),
                'Base WR%'  : round(base_wr * 100, 1),
                'Threshold' : None,
                'Trades'    : 0,
                'Wins'      : 0,
                'Win%'      : None,
                'Best C'    : best_C_str,
                'Train AUC' : round(train_auc, 3),
                'Skipped'   : True,
            })
            print(f'  {yr}-{mo:02d} | train={len(tr):3d}d | SKIPPED '
                  f'(AUC={train_auc:.3f} std={prob_std:.4f}) | C={best_C_str}')

        else:
            # Auto-select threshold on TRAINING data only
            tr_tmp         = tr.copy()
            tr_tmp['prob'] = probs_tr
            thr            = auto_threshold(tr_tmp, base_wr)

            for j, (_, row) in enumerate(te.iterrows()):
                p = float(probs_te[j])
                oos_rows.append({
                    'date'       : row['date'],
                    'month'      : test_month,
                    'model'      : model_name,
                    'prob'       : p,
                    'threshold'  : thr,
                    'traded'     : p >= thr,
                    'win'        : bool(row['win']),
                    'exit_reason': row['exit_reason'],
                    'entry_prem' : row['entry_prem'],
                    'exit_prem'  : row['exit_prem'],
                    'dte'        : row['dte'],
                    'base_wr'    : base_wr,
                    'VIX_INDIA_level': row['VIX_INDIA_level'],
                })

            n_traded = int((probs_te >= thr).sum())
            wins_m   = sum(1 for j in range(len(te)) if probs_te[j] >= thr and te['win'].iloc[j])

            monthly.append({
                'Month'     : f'{yr}-{mo:02d}',
                'Train days': len(tr),
                'Test days' : len(te),
                'Base WR%'  : round(base_wr * 100, 1),
                'Threshold' : thr,
                'Trades'    : n_traded,
                'Wins'      : wins_m,
                'Win%'      : round(wins_m / n_traded * 100, 1) if n_traded > 0 else None,
                'Best C'    : best_C_str,
                'Train AUC' : round(train_auc, 3),
                'Skipped'   : False,
            })
            print(f'  {yr}-{mo:02d} | train={len(tr):3d}d | thr={thr:.2f} | '
                  f'trades={n_traded} wins={wins_m} | AUC={train_auc:.3f} | C={best_C_str}')

    return oos_rows, monthly


print('Walk-forward engine ready.')
print(f'Will test {len(all_months) - oos_start_idx} months OOS ({OOS_FIRST_MONTH} → {all_months[-1]})')

Walk-forward engine ready.
Will test 15 months OOS ((2025, 1) → (2026, 3))


In [21]:
# ── Run L1 Logistic walk-forward ──────────────────────────────────────────────
print('=== L1 LOGISTIC REGRESSION — WALK-FORWARD ===')
print(f'C grid: {CV_C_GRID}  |  class_weight: {CLASS_WEIGHT}  |  5-fold CV (roc_auc)\n')

def fit_logistic(Xs_tr, y_tr):
    m = LogisticRegressionCV(
        Cs=CV_C_GRID, penalty='l1', solver='liblinear',
        class_weight=CLASS_WEIGHT, cv=5, scoring='roc_auc',
        max_iter=1000, random_state=42,
    )
    m.fit(Xs_tr, y_tr)
    return m

logit_oos_rows, logit_monthly = run_walk_forward('L1-Logistic', fit_logistic)
logit_monthly_df = pd.DataFrame(logit_monthly)

print(f'\n--- Monthly OOS breakdown ---')
print(logit_monthly_df.to_string(index=False))

logit_ledger = compute_ledger(logit_oos_rows, use_kelly=False)
print_ledger_summary(logit_ledger, 'L1 Logistic — Walk-Forward OOS (Jan 2025 – Mar 2026)')

=== L1 LOGISTIC REGRESSION — WALK-FORWARD ===
C grid: [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]  |  class_weight: {0: 1.0, 1: 2.5}  |  5-fold CV (roc_auc)

  2025-01 | train=170d | SKIPPED (AUC=0.500 std=0.0000) | C=0.01
  2025-02 | train=187d | SKIPPED (AUC=0.500 std=0.0000) | C=0.01
  2025-03 | train=200d | SKIPPED (AUC=0.500 std=0.0000) | C=0.01
  2025-04 | train=215d | SKIPPED (AUC=0.500 std=0.0000) | C=0.01
  2025-05 | train=224d | SKIPPED (AUC=0.500 std=0.0000) | C=0.01
  2025-06 | train=240d | thr=0.62 | trades=0 wins=0 | AUC=0.663 | C=0.20
  2025-07 | train=254d | thr=0.68 | trades=0 wins=0 | AUC=0.667 | C=0.50
  2025-08 | train=273d | thr=0.68 | trades=0 wins=0 | AUC=0.660 | C=0.50
  2025-09 | train=286d | thr=0.64 | trades=1 wins=0 | AUC=0.659 | C=0.20
  2025-10 | train=302d | thr=0.58 | trades=0 wins=0 | AUC=0.643 | C=0.10
  2025-11 | train=314d | thr=0.58 | trades=0 wins=0 | AUC=0.644 | C=0.10
  2025-12 | train=328d | SKIPPED (AUC=0.500 std=0.0000) | C=0.01
  2026-01 | tra

In [22]:
# ── Run Random Forest walk-forward ────────────────────────────────────────────
print('=== RANDOM FOREST — WALK-FORWARD ===')
print(f'max_depth={RF_MAX_DEPTH}  min_samples_leaf={RF_MIN_SAMPLES_LEAF}  n_estimators={RF_N_ESTIMATORS}\n')

def fit_rf(Xs_tr, y_tr):
    m = RandomForestClassifier(
        n_estimators     = RF_N_ESTIMATORS,
        max_depth        = RF_MAX_DEPTH,
        min_samples_leaf = RF_MIN_SAMPLES_LEAF,
        class_weight     = CLASS_WEIGHT,
        random_state     = 42,
        n_jobs           = -1,
    )
    m.fit(Xs_tr, y_tr)
    return m

rf_oos_rows, rf_monthly = run_walk_forward('RandomForest', fit_rf)
rf_monthly_df = pd.DataFrame(rf_monthly)

print(f'\n--- Monthly OOS breakdown ---')
print(rf_monthly_df[['Month','Train days','Test days','Base WR%','Threshold','Trades','Wins','Win%']].to_string(index=False))

rf_ledger = compute_ledger(rf_oos_rows, use_kelly=False)
print_ledger_summary(rf_ledger, 'Random Forest — Walk-Forward OOS (Jan 2025 – Mar 2026)')

# Feature importance from the last fitted RF (largest training window)
last_rf = fit_rf(
    StandardScaler().fit_transform(merged[merged['month_key'] < all_months[-1]][FEATURES].values),
    merged[merged['month_key'] < all_months[-1]]['win'].astype(int).values,
)
fi_df = (pd.DataFrame({'Feature': FEATURES, 'Importance': last_rf.feature_importances_})
         .sort_values('Importance', ascending=False))
print('\nRF Feature Importance (trained on all data except Mar 2026):')
print(fi_df.to_string(index=False))

=== RANDOM FOREST — WALK-FORWARD ===
max_depth=3  min_samples_leaf=20  n_estimators=300

  2025-01 | train=170d | thr=0.54 | trades=3 wins=0 | AUC=0.824 | C=N/A
  2025-02 | train=187d | thr=0.54 | trades=0 wins=0 | AUC=0.842 | C=N/A
  2025-03 | train=200d | thr=0.58 | trades=0 wins=0 | AUC=0.848 | C=N/A
  2025-04 | train=215d | thr=0.56 | trades=0 wins=0 | AUC=0.854 | C=N/A
  2025-05 | train=224d | thr=0.56 | trades=0 wins=0 | AUC=0.858 | C=N/A
  2025-06 | train=240d | thr=0.56 | trades=0 wins=0 | AUC=0.863 | C=N/A
  2025-07 | train=254d | thr=0.56 | trades=0 wins=0 | AUC=0.859 | C=N/A
  2025-08 | train=273d | thr=0.56 | trades=0 wins=0 | AUC=0.864 | C=N/A
  2025-09 | train=286d | thr=0.56 | trades=0 wins=0 | AUC=0.871 | C=N/A
  2025-10 | train=302d | thr=0.54 | trades=0 wins=0 | AUC=0.879 | C=N/A
  2025-11 | train=314d | thr=0.54 | trades=0 wins=0 | AUC=0.869 | C=N/A
  2025-12 | train=328d | thr=0.54 | trades=0 wins=0 | AUC=0.864 | C=N/A
  2026-01 | train=343d | thr=0.54 | trades=0 wi

In [23]:
# ── Model comparison ──────────────────────────────────────────────────────────
logit_monthly_df = logit_monthly_df.rename(columns={'Trades': 'L_Trades', 'Wins': 'L_Wins', 'Win%': 'L_Win%', 'Threshold': 'L_Thr'})
rf_monthly_df2   = rf_monthly_df.rename(columns={'Trades': 'RF_Trades', 'Wins': 'RF_Wins', 'Win%': 'RF_Win%', 'Threshold': 'RF_Thr'})
compare = logit_monthly_df[['Month', 'Base WR%', 'L_Thr', 'L_Trades', 'L_Wins', 'L_Win%']].merge(
    rf_monthly_df2[['Month', 'RF_Thr', 'RF_Trades', 'RF_Wins', 'RF_Win%']], on='Month', how='outer'
)

print('Monthly OOS comparison — L1 Logistic vs Random Forest:')
print('(Win% = win rate ONLY on traded days — NaN = no trades that month)')
print(compare.to_string(index=False))

# Agreement analysis
logit_traded_dates = {r['date'] for r in logit_oos_rows if r['traded']}
rf_traded_dates    = {r['date'] for r in rf_oos_rows    if r['traded']}
both_dates         = logit_traded_dates & rf_traded_dates
only_logit         = logit_traded_dates - rf_traded_dates
only_rf            = rf_traded_dates - logit_traded_dates

print(f'\nSignal agreement:')
print(f'  Both models trade     : {len(both_dates)} days')
print(f'  Logistic only         : {len(only_logit)} days')
print(f'  RF only               : {len(only_rf)} days')

if both_dates:
    both_wins = sum(1 for r in logit_oos_rows if r['date'] in both_dates and r['win'])
    both_wr   = both_wins / len(both_dates)
    print(f'  Win rate (both agree) : {both_wr:.1%}  ({both_wins}/{len(both_dates)} wins)')

if only_logit:
    ol_wins = sum(1 for r in logit_oos_rows if r['date'] in only_logit and r['win'])
    print(f'  Win rate (logit only) : {ol_wins/len(only_logit):.1%}  ({ol_wins}/{len(only_logit)} wins)')

if only_rf:
    rf_wins = sum(1 for r in rf_oos_rows if r['date'] in only_rf and r['win'])
    print(f'  Win rate (RF only)    : {rf_wins/len(only_rf):.1%}  ({rf_wins}/{len(only_rf)} wins)')

Monthly OOS comparison — L1 Logistic vs Random Forest:
(Win% = win rate ONLY on traded days — NaN = no trades that month)
  Month  Base WR%  L_Thr  L_Trades  L_Wins  L_Win%  RF_Thr  RF_Trades  RF_Wins  RF_Win%
2025-01      26.5    NaN         0       0     NaN    0.54          3        0      0.0
2025-02      25.7    NaN         0       0     NaN    0.54          0        0      NaN
2025-03      27.0    NaN         0       0     NaN    0.58          0        0      NaN
2025-04      26.0    NaN         0       0     NaN    0.56          0        0      NaN
2025-05      25.4    NaN         0       0     NaN    0.56          0        0      NaN
2025-06      25.0   0.62         0       0     NaN    0.56          0        0      NaN
2025-07      24.4   0.68         0       0     NaN    0.56          0        0      NaN
2025-08      24.9   0.68         0       0     NaN    0.56          0        0      NaN
2025-09      24.8   0.64         1       0     0.0    0.56          0        0      Na

In [24]:
# ── Bootstrap confidence intervals ────────────────────────────────────────────
# Resamples the OOS trades (with replacement) to estimate ROI distribution.
# Does NOT maintain compounding sequence — treats each trade as independent draw.
# This is standard for small-sample financial backtests.

def bootstrap_ci(ledger: pd.DataFrame, label: str, n: int = N_BOOTSTRAP):
    if ledger.empty:
        print(f'\n{label}: no trades — skipping.')
        return

    pnls        = ledger['Trade PnL'].values
    actual_roi  = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    n_trades    = len(pnls)

    boot_rois = np.array([
        np.random.choice(pnls, size=n_trades, replace=True).sum() / STARTING_CAPITAL * 100
        for _ in range(n)
    ])

    p5, p25, p50, p75, p95 = np.percentile(boot_rois, [5, 25, 50, 75, 95])
    p_pos = (boot_rois > 0).mean()

    print(f'\n{"="*58}')
    print(f'  Bootstrap CI — {label}')
    print(f'  ({n_trades} trades, {n:,} resamples, non-compounding)')
    print(f'{"="*58}')
    print(f'  Actual OOS ROI   : {actual_roi:+.1f}%')
    print(f'  Bootstrap median : {p50:+.1f}%')
    print(f'  90% CI           : [{p5:+.1f}%,  {p95:+.1f}%]')
    print(f'  50% CI           : [{p25:+.1f}%,  {p75:+.1f}%]')
    print(f'  P(ROI > 0)       : {p_pos:.1%}')
    print(f'{"="*58}')
    if p_pos >= 0.70 and p5 > -30:
        print(f'  --> Edge plausible: P(positive)={p_pos:.0%}, lower 90% CI={p5:+.1f}%')
    elif p_pos >= 0.55:
        print(f'  --> Weak edge: P(positive)={p_pos:.0%} — need more trades to confirm')
    else:
        print(f'  --> Edge not confirmed: P(positive)={p_pos:.0%}')


bootstrap_ci(logit_ledger, 'L1 Logistic')
bootstrap_ci(rf_ledger,    'Random Forest')

# Combined: if both models agree on a day, use that trade once
if both_dates:
    both_rows = [r for r in logit_oos_rows if r['date'] in both_dates]
    both_ledger = compute_ledger(both_rows, use_kelly=False)
    bootstrap_ci(both_ledger, 'Ensemble (both models agree)')


  Bootstrap CI — L1 Logistic
  (7 trades, 10,000 resamples, non-compounding)
  Actual OOS ROI   : -19.5%
  Bootstrap median : -20.3%
  90% CI           : [-63.0%,  +30.3%]
  50% CI           : [-39.3%,  +0.1%]
  P(ROI > 0)       : 25.1%
  --> Edge not confirmed: P(positive)=25%

  Bootstrap CI — Random Forest
  (6 trades, 10,000 resamples, non-compounding)
  Actual OOS ROI   : -28.6%
  Bootstrap median : -29.6%
  90% CI           : [-57.5%,  +4.8%]
  50% CI           : [-41.8%,  -15.8%]
  P(ROI > 0)       : 7.0%
  --> Edge not confirmed: P(positive)=7%

  Bootstrap CI — Ensemble (both models agree)
  (3 trades, 10,000 resamples, non-compounding)
  Actual OOS ROI   : -4.1%
  Bootstrap median : -4.1%
  90% CI           : [-23.2%,  +15.0%]
  50% CI           : [-18.4%,  +10.2%]
  P(ROI > 0)       : 36.8%
  --> Edge not confirmed: P(positive)=37%


In [25]:
# ── Kelly criterion lot sizing ─────────────────────────────────────────────────
# Kelly formula: f* = p - (1-p)/b  where b = TP/SL = 2.667
# Kelly lots = f* * capital / (entry_prem * LOT_SIZE)

print('Kelly Criterion Analysis')
print('='*60)
print(f'Odds b = TP/SL = {TP_PCT}/{SL_PCT} = {KELLY_ODDS:.3f}')
print(f'Formula: f* = p - (1-p)/{KELLY_ODDS:.3f}')
print(f'At breakeven ({BREAKEVEN:.1%}): f* = {max(0, BREAKEVEN - (1-BREAKEVEN)/KELLY_ODDS):.3f}  (no bet)')
print(f'At 35% win:   f* = {max(0, 0.35 - 0.65/KELLY_ODDS):.3f}  ({max(0, 0.35 - 0.65/KELLY_ODDS)*100:.1f}% of capital)')
print(f'At 40% win:   f* = {max(0, 0.40 - 0.60/KELLY_ODDS):.3f}  ({max(0, 0.40 - 0.60/KELLY_ODDS)*100:.1f}% of capital)')
print(f'At 50% win:   f* = {max(0, 0.50 - 0.50/KELLY_ODDS):.3f}  ({max(0, 0.50 - 0.50/KELLY_ODDS)*100:.1f}% of capital)')
print()
print('Fixed sizing vs Kelly sizing at different capital levels (Rs 100 prem):')
print(f'{"Capital":>12} | {"Fixed lots":>10} | {"Kelly (40%wr)":>13} | {"Half-Kelly":>10}')
print('-' * 55)
f40 = max(0, 0.40 - 0.60 / KELLY_ODDS)
for cap in [200_000, 300_000, 400_000, 500_000, 600_000]:
    cost_lot   = 100 * LOT_SIZE
    fixed_lots = min(MAX_LOTS, max(BASE_LOTS, int(cap // cost_lot)))
    kelly_lots = min(MAX_LOTS, max(1, int(f40 * cap / cost_lot)))
    hkelly_lots= min(MAX_LOTS, max(1, int(0.5 * f40 * cap / cost_lot)))
    print(f'Rs {cap:>9,} | {fixed_lots:>10} | {kelly_lots:>13} | {hkelly_lots:>10}')
print()
print('Observation: Fixed sizing hits MAX_LOTS=25 cap early.')
print('Kelly scales proportionally — never caps if model has consistent 40% edge.')
print()

# Compare fixed vs Kelly on actual OOS trades
for label, oos_rows in [('L1 Logistic', logit_oos_rows), ('Random Forest', rf_oos_rows)]:
    fixed_ledger = compute_ledger(oos_rows, use_kelly=False)
    kelly_ledger = compute_ledger(oos_rows, use_kelly=True)
    if fixed_ledger.empty:
        continue
    fixed_roi = (fixed_ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    kelly_roi = (kelly_ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100 if not kelly_ledger.empty else 0
    fixed_dd  = fixed_ledger['DD%'].max()
    kelly_dd  = kelly_ledger['DD%'].max() if not kelly_ledger.empty else 0
    print(f'{label}:')
    print(f'  Fixed lot sizing : ROI {fixed_roi:+.1f}%  MaxDD {fixed_dd:.1f}%')
    print(f'  Kelly lot sizing : ROI {kelly_roi:+.1f}%  MaxDD {kelly_dd:.1f}%')
    print()
    # Show lot comparison per trade
    if not fixed_ledger.empty and not kelly_ledger.empty:
        comp = fixed_ledger[['Date','P(win)','Entry','Lots']].rename(columns={'Lots':'Fixed Lots'}).merge(
            kelly_ledger[['Date','Lots']].rename(columns={'Lots':'Kelly Lots'}), on='Date', how='outer'
        )
        print(f'  Trade-level lot comparison:')
        print(comp.to_string(index=False))
        print()

Kelly Criterion Analysis
Odds b = TP/SL = 0.4/0.15 = 2.667
Formula: f* = p - (1-p)/2.667
At breakeven (27.3%): f* = 0.000  (no bet)
At 35% win:   f* = 0.106  (10.6% of capital)
At 40% win:   f* = 0.175  (17.5% of capital)
At 50% win:   f* = 0.312  (31.2% of capital)

Fixed sizing vs Kelly sizing at different capital levels (Rs 100 prem):
     Capital | Fixed lots | Kelly (40%wr) | Half-Kelly
-------------------------------------------------------
Rs   200,000 |         25 |             4 |          2
Rs   300,000 |         25 |             7 |          3
Rs   400,000 |         25 |             9 |          4
Rs   500,000 |         25 |            11 |          5
Rs   600,000 |         25 |            14 |          7

Observation: Fixed sizing hits MAX_LOTS=25 cap early.
Kelly scales proportionally — never caps if model has consistent 40% edge.

L1 Logistic:
  Fixed lot sizing : ROI -19.5%  MaxDD 24.8%
  Kelly lot sizing : ROI +1.5%  MaxDD 8.6%

  Trade-level lot comparison:
      Date 

In [26]:
# ── Final summary ──────────────────────────────────────────────────────────────
def _row(ledger, label):
    if ledger.empty:
        return {'Strategy': label, 'Trades': 0, 'Win%': 'N/A', 'ROI': 'N/A', 'MaxDD': 'N/A', 'Note': ''}
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    roi   = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['DD%'].max()
    return {'Strategy': label, 'Trades': total,
            'Win%': f'{wins/total*100:.1f}%', 'ROI': f'{roi:+.1f}%', 'MaxDD': f'{maxdd:.1f}%', 'Note': ''}

rows = [
    {'Strategy': 'v4.3  (in-sample, Jan 2024–Mar 2026)',       'Trades': 57,  'Win%': '38.6%', 'ROI': '+145.8%', 'MaxDD': '52.1%', 'Note': 'IN-SAMPLE BIAS'},
    {'Strategy': 'v7    (single split OOS Jul 2025–Mar 2026)', 'Trades': 10,  'Win%': '40.0%', 'ROI':   '+8.4%', 'MaxDD': '21.0%', 'Note': 'threshold in-sample'},
    _row(logit_ledger, 'v8    L1 Logistic walk-forward OOS'),
    _row(rf_ledger,    'v8    Random Forest walk-forward OOS'),
]

rows[-2]['Note'] = 'monthly retrain + threshold'
rows[-1]['Note'] = 'monthly retrain + threshold'

summary_df = pd.DataFrame(rows)

print()
print('=' * 80)
print('  v8 FINAL SUMMARY')
print('=' * 80)
print(summary_df.to_string(index=False))
print('=' * 80)
print()
print(f'Walk-forward period : Jan 2025 – Mar 2026  ({len(all_months) - oos_start_idx} months, expanding window)')
print(f'Warmup              : Jan 2024 – Dec 2024  ({oos_start_idx} months, ~{oos_start_idx*17} days)')
print(f'Breakeven win rate  : {BREAKEVEN:.1%}')
print()
print('Bootstrap CI and Kelly analysis: see Cells 8 and 9.')
print()
print('Key questions to answer from these results:')
print('  1. Does walk-forward OOS ROI stay positive? (less optimistic than v7 single split)')
print('  2. Does RF outperform logistic? (non-linearity worth the variance?)')
print('  3. Is P(ROI>0) from bootstrap >= 70%? (edge plausible?)')
print('  4. Does Kelly sizing improve or worsen the risk-adjusted return?')


  v8 FINAL SUMMARY
                                  Strategy  Trades  Win%     ROI MaxDD                        Note
      v4.3  (in-sample, Jan 2024–Mar 2026)      57 38.6% +145.8% 52.1%              IN-SAMPLE BIAS
v7    (single split OOS Jul 2025–Mar 2026)      10 40.0%   +8.4% 21.0%         threshold in-sample
        v8    L1 Logistic walk-forward OOS       7 28.6%  -19.5% 24.8% monthly retrain + threshold
      v8    Random Forest walk-forward OOS       6 16.7%  -28.6% 33.8% monthly retrain + threshold

Walk-forward period : Jan 2025 – Mar 2026  (15 months, expanding window)
Warmup              : Jan 2024 – Dec 2024  (12 months, ~204 days)
Breakeven win rate  : 27.3%

Bootstrap CI and Kelly analysis: see Cells 8 and 9.

Key questions to answer from these results:
  1. Does walk-forward OOS ROI stay positive? (less optimistic than v7 single split)
  2. Does RF outperform logistic? (non-linearity worth the variance?)
  3. Is P(ROI>0) from bootstrap >= 70%? (edge plausible?)
  4. D